In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os, cv2, random
import pandas as pd

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import Convolution2D,MaxPooling2D, Dense, Flatten, InputLayer
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.backend import clear_session

In [2]:
data_dir = "../data"
train_dir = data_dir+"/train"
test_dir = data_dir+"/test"

In [3]:
df = pd.DataFrame()
fnames = os.listdir(train_dir)

class_name = []
for name in fnames:
    class_name.append(name.split('.')[0])
    
data = {'filename':fnames,'class':class_name}
df = pd.DataFrame(data)
df = df.sample(frac=1)

train_datagen = ImageDataGenerator(rescale = 1/255,
                                   rotation_range=20,
                                   shear_range=0.2,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   horizontal_flip=True,
                                   zoom_range=0.2
                                  )
valid_datagen = ImageDataGenerator(rescale = 1/255)

idx = int(0.8*len(df))
train_df = df.iloc[:idx]
valid_df = df.iloc[idx:]

target = (100,100)

train_set = train_datagen.flow_from_dataframe(train_df,
                                              directory=train_dir,
                                              shuffle=True,
                                              target_size = target,
                                              batch_size = 64,
                                              class_mode = 'binary')

valid_set = valid_datagen.flow_from_dataframe(valid_df,
                                              directory=train_dir,
                                              shuffle=False,
                                              target_size = target,
                                              batch_size = 32,
                                              class_mode = 'binary')

Found 20000 validated image filenames belonging to 2 classes.
Found 5000 validated image filenames belonging to 2 classes.


In [ ]:
clear_session()

model = Sequential([
    InputLayer(input_shape=target+(3,)),
    Convolution2D(16,3,activation='relu'),
    MaxPooling2D(2),
    Convolution2D(32,3,activation='relu'),
    MaxPooling2D(2),
    Convolution2D(64,3,activation='relu'),
    MaxPooling2D(2),
    Flatten(),
    Dense(512,activation='relu'),
    Dense(1,activation='sigmoid')
])
model.summary()

opt = SGD(learning_rate = 0.05,
          momentum = 0.9,
          nesterov = True)
model.compile(loss='binary_crossentropy',optimizer=opt,metrics=['acc'])
model.optimizer.get_config()

In [14]:
clear_session()

model = VGG16(include_top=False, input_shape=target+(3,))
for layer in model.layers:
    layer.trainable = False
flat1 = Flatten()(model.layers[-1].output)
class1 = Dense(128, activation='relu', kernel_initializer='he_uniform')(flat1)
output = Dense(1, activation='sigmoid')(class1)
model = Model(inputs=model.inputs, outputs=output)

model.summary()

opt = SGD(learning_rate = 0.001,
          momentum = 0.9)
model.compile(loss='binary_crossentropy',optimizer=opt,metrics=['acc'])
model.optimizer.get_config()

Found 20000 validated image filenames belonging to 2 classes.
Found 5000 validated image filenames belonging to 2 classes.
Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_1 (InputLayer)         [(None, 244, 244, 3)]     0         
_________________________________________________________________
block1_conv1 (Conv2D)        (None, 244, 244, 64)      1792      
_________________________________________________________________
block1_conv2 (Conv2D)        (None, 244, 244, 64)      36928     
_________________________________________________________________
block1_pool (MaxPooling2D)   (None, 122, 122, 64)      0         
_________________________________________________________________
block2_conv1 (Conv2D)        (None, 122, 122, 128)     73856     
_________________________________________________________________
block2_conv2 (Conv2D)        (None, 122, 122, 128)     147584    
____

{'name': 'SGD',
 'learning_rate': 0.001,
 'decay': 0.0,
 'momentum': 0.9,
 'nesterov': False}

In [15]:
history = model.fit(train_set,
                    validation_data=valid_set,
                    steps_per_epoch=train_set.n//train_set.batch_size,
                    validation_steps=valid_set.n//valid_set.batch_size,
                    epochs = 10)

Epoch 1/10
131/312 [===========>..................] - ETA: 39:48 - loss: 0.5503 - acc: 0.7055

KeyboardInterrupt: 

In [ ]:
legend = ['train','validation']

plt.plot(history.history['acc'])
plt.plot(history.history['val_acc'])
plt.title("Accuracy")
plt.xlabel("epochs")
plt.ylabel("acc")
plt.legend(legend,loc='upper left')
plt.show()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title("Binary cross-entropy loss")
plt.xlabel("epochs")
plt.ylabel("loss")
plt.legend(legend,loc='upper left')
plt.show()

In [ ]:
model.save('my_model-3_block-aug-20_epoch.h5')

In [ ]:
model = load_model('my_model-3_block-aug-50_epoch')

In [ ]:
print("Accuracy:",model.evaluate(valid_set))

In [ ]:
layer_outputs = []
for layer in model.layers:
    if 'conv' not in(layer.name):
        continue
    layer_outputs.append(layer.output)

activation_model = Model(inputs = model.input, outputs = layer_outputs)

In [ ]:
def preprocess(img):
    img = cv2.resize(img,target)
    img = img/255
    return np.array(img)
    
def predict(img):
    img = preprocess(img)
    img = img.reshape((1,)+img.shape)
    probability = model.predict(img)
    return probability

def getLabel(probability):
    if probability<0.5:
        probability=0
    else:
        probability=1
    return list(train_set.class_indices)[probability]

def visualize(img):
    img = preprocess(img)
    img = img.reshape((1,)+img.shape)
    fmaps = activation_model.predict(img)

    for i in range(len(fmaps)):
        activation = fmaps[i]
        
        fig = plt.figure(figsize=(20,15))
        fig.suptitle(layer_outputs[i].name)
        
        for j in range(min(8*8,activation.shape[-1])):
            plt.subplot(8,8,j+1)
            plt.imshow(activation[0,:,:,j])
    plt.show()

WIN_SIZES=[]
for i in range(100,220,20):
    WIN_SIZES.append(i)
    
def get_box(img,step=20,win_sizes=WIN_SIZES):
    box_list = []
    raw_prob = predict(img)
    if (raw_prob<0.5):
        raw_prob=0
    else:
        raw_prob=1
        
    for win in win_sizes:
        print("Run with window size:",str(win))
        for top in range(0,img.shape[0]-win+1,step):
            for left in range(0,img.shape[1]-win+1,step):
                box = (left,top,left+win,top+win)
                crop = img[box[1]:box[3],box[0]:box[2]]
                prob = predict(crop)
                distance = abs(raw_prob-prob)
                box_list.append((distance,box))
    return box_list

In [ ]:
test_fnames = os.listdir(test_dir)
random.shuffle(test_fnames)
result = []

nPic = 10

for fnames in test_fnames:
    pred_path = os.path.join(test_dir,fnames)
    img = cv2.imread(pred_path)
    img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    
    title = getLabel(predict(img))
    plt.imshow(img)
    plt.title(title)
    plt.show()
    
    nPic -= 1
    if (nPic==0):
        break

In [ ]:
test_fnames = os.listdir(test_dir)
idPic = 1
pred_path = os.path.join(test_dir,test_fnames[idPic])
img = cv2.imread(pred_path)
img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

title = getLabel(predict(img))
        
plt.imshow(img)
plt.title(title)
plt.show()

# visualize(img)

def func(x):
    return x[0]

box_list, prob = get_box(img)
box_list.sort(key=func)
for tup in box_list[:3]:
    box = tup[1]
    startPoint = (box[0],box[1])
    endPoint = (box[2],box[3])

    COLOR=(255,0,0)
    img = cv2.rectangle(img,startPoint,endPoint,COLOR,2)
plt.imshow(img)